In [19]:
#Common Table Expressions(WITH clause).

In [20]:
#CTE ek temporary named result set hoti hai jo query ke start me WITH keyword se define hoti hai - yeh subquery (jo hume day15 me seekha) ka hi ek cleaner,jyada readable version hai. Bade complex queries me nested subqueries padhna difficult ho jaata hai, CTE se code readable or reusable ban jata hai.

In [21]:
#01:Basic CTE - day 15 ka scaler subquery example, CTE se rewrite karke:

In [23]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/db/ecommerce.db')

# Subquery version (Day 15 wala):
# SELECT ... WHERE total_spend > (SELECT AVG(...) FROM (SELECT ...))

# CTE version — same logic, zyada readable:
q1 = pd.read_sql("""
    WITH customer_totals AS (
        SELECT "Customer ID", SUM(OrderValue) as total_spend
        FROM orders
        GROUP BY "Customer ID"
    )
    SELECT "Customer ID", total_spend
    FROM customer_totals
    WHERE total_spend > (SELECT AVG(total_spend) FROM customer_totals)
    ORDER BY total_spend DESC
    LIMIT 10
""", conn)
print(q1)

   Customer ID  total_spend
0      18102.0    598215.22
1      14646.0    523342.07
2      14156.0    296564.69
3      14911.0    270248.53
4      17450.0    233579.39
5      13694.0    190825.52
6      17511.0    171885.98
7      12415.0    143269.29
8      16684.0    141502.25
9      15061.0    136391.48


In [24]:
#customer_totals CTE ko humne 2 baar use kiya (ek FROM me, ek subquery me) - bina isse dubara likhe. yeh CTE ka bada fayda hai-reusability.

In [26]:
#02:Multiple CTEs ek saath (comma se separate karke):

In [25]:
q2 = pd.read_sql("""
    WITH customer_totals AS (
        SELECT "Customer ID", SUM(OrderValue) as total_spend
        FROM orders
        GROUP BY "Customer ID"
    ),
    country_avg AS (
        SELECT c.Country, AVG(ct.total_spend) as avg_country_spend
        FROM customer_totals ct
        INNER JOIN customers c ON ct."Customer ID" = c."Customer ID"
        GROUP BY c.Country
    )
    SELECT * FROM country_avg
    ORDER BY avg_country_spend DESC
    LIMIT 10
""", conn)
print(q2)

       Country  avg_country_spend
0         EIRE      115845.096000
1  Netherlands       23848.910870
2    Singapore       13158.160000
3    Australia       11482.080000
4    Lithuania        6553.740000
5      Denmark        6078.402500
6      Iceland        5633.320000
7       Sweden        4602.916842
8  Switzerland        4592.218636
9        Japan        4377.658000


In [27]:
#03:CTE ke andar window function bhi use ho sakta hai (day 12 ka top-3-per-country example, ab CTE se cleaner):

In [28]:
q3 = pd.read_sql("""
    WITH ranked_customers AS (
        SELECT c."Customer ID", c.Country, SUM(o.OrderValue) as total_spend,
               ROW_NUMBER() OVER (PARTITION BY c.Country ORDER BY SUM(o.OrderValue) DESC) as country_rank
        FROM customers c
        INNER JOIN orders o ON c."Customer ID" = o."Customer ID"
        GROUP BY c."Customer ID", c.Country
    )
    SELECT * FROM ranked_customers
    WHERE country_rank <= 3
    ORDER BY Country, country_rank
""", conn)
print(q3)

    Customer ID         Country  total_spend  country_rank
0       12415.0       Australia    143269.29             1
1       12431.0       Australia     10719.41             2
2       12422.0       Australia      4119.35             3
3       12429.0         Austria      7435.51             1
4       12370.0         Austria      4320.31             2
..          ...             ...          ...           ...
93      13694.0  United Kingdom    190825.52             3
94      16320.0     Unspecified      4428.85             1
95      14265.0     Unspecified      1373.35             2
96      12363.0     Unspecified       552.00             3
97      18140.0     West Indies       536.41             1

[98 rows x 4 columns]


In [29]:
#compare karo isko day 12 ke nested subquery version se - CTE jyada clean padhta hai kyuki logic top-se-bottom flow karta hai, nested brackets nahi hote.

In [30]:
#practice questions.

In [31]:
#1.Ek CTE banao jo har customer ka total_spend aur total_orders nikale, phir usi CTE ko use karke un customers ko dhoondo jinka total_orders > 5 hai.

In [32]:
#

In [33]:
#2.Day 14 wala "recency gap" wala query CTE se rewrite karo — pehle CTE mein gaps calculate karo, phir outer query mein average nikalo.

In [34]:
#

In [35]:
conn.close()